# Karaokê: processamento no Colab (GPU)

Este notebook baixa as músicas do YouTube, separa a voz do instrumental com IA (GPU), busca a letra
sincronizada e gera um **pacote .zip** para importar na aplicação de karaokê que roda no seu Mac.

**Antes de rodar:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.

Como importar no Mac depois:
1. Copie o `.zip` para a pasta `data/import/` do projeto.
2. No celular do anfitrião, aba com o seu nome → **Importar pacotes**. Ou, com o servidor parado:
   `uv run python -m karaoke.importer data/import/karaoke-pack-XXXX.zip`.


In [ ]:
#@title 1. Instalação (≈2 min)
!pip -q install -U yt-dlp "audio-separator[gpu]" httpx
!nvidia-smi -L || echo "⚠️ Sem GPU: ative em Ambiente de execução → Alterar tipo → GPU"
!ffmpeg -version | head -1


In [ ]:
#@title 2. Músicas e opções
# Uma música por linha: link do YouTube, ID do vídeo ou texto de busca.
# Prefixo "karaoke:" marca vídeos que JÁ são versão karaokê (não separa a voz).
SONGS = """
https://www.youtube.com/watch?v=qHJ8CIM4jCw
karaoke: Sing King Evidências Chitãozinho e Xororó karaoke
Legião Urbana Pais e Filhos
"""

# BS-Roformer-Viperx-1297 é o modelo com melhor qualidade de instrumental (SDR 16.5) na lista do audio-separator.
MODEL = "model_bs_roformer_ep_317_sdr_12.9755.ckpt"  #@param ["model_bs_roformer_ep_317_sdr_12.9755.ckpt", "model_bs_roformer_ep_368_sdr_12.9628.ckpt", "melband_roformer_instvox_duality_v2.ckpt", "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt", "UVR-MDX-NET-Inst_HQ_4.onnx"]
MAX_HEIGHT = 1080  #@param [720, 1080]
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
# Se o YouTube bloquear o download no Colab ("Sign in to confirm you're not a bot"), exporte os
# cookies do seu navegador (extensão "Get cookies.txt LOCALLY"), envie o arquivo para o Colab e informe o caminho:
COOKIES_FILE = ""  #@param {type:"string"}


In [ ]:
#@title 3. Processamento
import json, re, time, unicodedata, subprocess, shutil, logging
from pathlib import Path
import httpx, yt_dlp
from audio_separator.separator import Separator

WORK = Path("/content/karaoke-work"); WORK.mkdir(exist_ok=True)
MODELS = Path("/content/models"); MODELS.mkdir(exist_ok=True)

# ---------- letras (lrclib.net) ----------
_NOISE = re.compile(r"[\(\[\{][^\)\]\}]*(official|oficial|video|vídeo|clipe|lyric|letra|audio|áudio|hd|4k|remaster|ao vivo|live|visualizer|legendado|karaok[eê]|instrumental|playback|cover)[^\)\]\}]*[\)\]\}]", re.I)
_FEAT = re.compile(r"\s*(feat\.?|ft\.?|part\.?|com)\s+.*$", re.I)
_SEP = re.compile(r"\s+[-–—|]\s+")
_LRC = re.compile(r"\[(\d+):(\d+(?:\.\d+)?)\]")

def clean_title(t):
    t = _NOISE.sub("", unicodedata.normalize("NFC", t))
    t = re.sub(r"\s*[\(\[\{]\s*[\)\]\}]", "", t); t = re.sub(r'["“”]', "", t)
    return re.sub(r"\s{2,}", " ", t).strip(" -–|")

def parse_lrc(text):
    lines = []
    for raw in text.splitlines():
        stamps = _LRC.findall(raw)
        if not stamps: continue
        content = _LRC.sub("", raw).strip()
        for mm, ss in stamps: lines.append({"t": int(mm) * 60 + float(ss), "text": content, "singer": None})
    return sorted(lines, key=lambda l: l["t"])

def fetch_lyrics(title, duration):
    queries = []
    parts = _SEP.split(clean_title(title), maxsplit=1)
    if len(parts) == 2: queries.append(f"{parts[0].strip()} {_FEAT.sub('', parts[1]).strip()}")
    c = clean_title(title)
    if c not in queries: queries.append(c)
    with httpx.Client(timeout=15, headers={"User-Agent": "karaoke-local/0.1 colab"}) as client:
        for q in queries:
            try:
                items = client.get("https://lrclib.net/api/search", params={"q": q}).json()
            except Exception:
                continue
            best = None
            for it in items:
                if not it.get("syncedLyrics") or it.get("instrumental"): continue
                score = abs(float(it["duration"]) - duration) if duration and it.get("duration") else 4.0
                if score > 8: continue
                if best is None or score < best[0]: best = (score, it)
            if best:
                lines = parse_lrc(best[1]["syncedLyrics"])
                if lines: return lines, f"lrclib: {best[1].get('artistName')} - {best[1].get('trackName')}"
    return None, "none"

# ---------- youtube ----------
def resolve(query):
    opts = {"quiet": True, "no_warnings": True, "skip_download": True, "noplaylist": True, "extract_flat": "in_playlist"}
    if COOKIES_FILE: opts["cookiefile"] = COOKIES_FILE
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/)([A-Za-z0-9_-]{11})", query)
    target = f"https://www.youtube.com/watch?v={m.group(1)}" if m else (f"https://www.youtube.com/watch?v={query}" if re.fullmatch(r"[A-Za-z0-9_-]{11}", query) else f"ytsearch1:{query}")
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(target, download=False)
    if "entries" in info: info = info["entries"][0]
    return info["id"]

def download(yt_id, dest):
    h = MAX_HEIGHT
    opts = {"quiet": True, "no_warnings": True, "noprogress": True, "noplaylist": True, "overwrites": True,
            "format": f"bestvideo[height<={h}][ext=mp4][vcodec^=avc1]+bestaudio[ext=m4a]/bestvideo[height<={h}][ext=mp4]+bestaudio[ext=m4a]/bestvideo[height<={h}]+bestaudio/best[height<={h}]/best",
            "merge_output_format": "mp4", "outtmpl": str(dest.with_suffix("")) + ".%(ext)s"}
    if COOKIES_FILE: opts["cookiefile"] = COOKIES_FILE
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(f"https://www.youtube.com/watch?v={yt_id}", download=True)
    return {"yt_id": yt_id, "title": unicodedata.normalize("NFC", info.get("title") or yt_id),
            "channel": info.get("channel") or info.get("uploader") or "", "duration": info.get("duration"),
            "thumbnail": f"https://i.ytimg.com/vi/{yt_id}/hqdefault.jpg"}

def ffmpeg(*args):
    subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", *args], check=True)

# ---------- separação ----------
separator = None
def separate(audio, out_dir):
    global separator
    if separator is None:
        separator = Separator(log_level=logging.WARNING, model_file_dir=str(MODELS), output_dir=str(out_dir), output_format="WAV")
        separator.load_model(model_filename=MODEL)
    separator.output_dir = str(out_dir)
    separator.model_instance.output_dir = str(out_dir)
    files = separator.separate(str(audio), custom_output_names={"Instrumental": "instrumental", "Vocals": "vocals"})
    return out_dir / "instrumental.wav", out_dir / "vocals.wav"

# ---------- loop ----------
jobs = [l.strip() for l in SONGS.strip().splitlines() if l.strip() and not l.strip().startswith("#")]
results = []
for raw in jobs:
    is_karaoke = raw.lower().startswith("karaoke:")
    query = raw.split(":", 1)[1].strip() if is_karaoke else raw
    t0 = time.time()
    try:
        yt_id = resolve(query)
        d = WORK / yt_id; d.mkdir(exist_ok=True)
        meta_path = d / "song.json"
        if meta_path.exists() and (d / "original.mp4").exists() and (is_karaoke or (d / "instrumental.mp4").exists()):
            print(f"✓ {yt_id} já processado, pulando"); results.append(yt_id); continue
        print(f"▶ {query[:60]} → {yt_id}")
        video = download(yt_id, d / "original.mp4"); print(f"   baixado: {video['title']} ({video['duration']}s)")
        if not is_karaoke:
            ffmpeg("-i", str(d / "original.mp4"), "-vn", "-ac", "2", "-ar", "44100", str(d / "audio.wav"))
            inst, voc = separate(d / "audio.wav", d)
            ffmpeg("-i", str(d / "original.mp4"), "-i", str(inst), "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", str(d / "instrumental.mp4"))
            ffmpeg("-i", str(voc), "-c:a", "aac", "-b:a", "160k", str(d / "vocals.m4a"))
            for f in (inst, voc, d / "audio.wav"): f.unlink(missing_ok=True)
            print("   voz separada")
        lyrics, source = fetch_lyrics(video["title"], video["duration"])
        print(f"   letra: {source}")
        song = {"video": video, "status": "ready", "progress": 1.0, "error": None,
                "has_original": True, "has_instrumental": not is_karaoke, "has_vocals": not is_karaoke,
                "lyrics": lyrics, "lyrics_source": source}
        meta_path.write_text(json.dumps(song, ensure_ascii=False, indent=1))
        results.append(yt_id)
        print(f"   ok em {time.time()-t0:.0f}s")
    except Exception as e:
        print(f"   ✗ erro: {type(e).__name__}: {str(e)[:200]}")

print(f"\n{len(results)} música(s) prontas: {results}")


In [ ]:
#@title 4. Gerar o pacote .zip
import zipfile, datetime
from pathlib import Path
name = f"karaoke-pack-{datetime.datetime.now():%Y%m%d-%H%M}.zip"
out = Path("/content") / name
with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as z:
    for yt_id in results:
        d = WORK / yt_id
        for f in ("song.json", "original.mp4", "instrumental.mp4", "vocals.m4a"):
            if (d / f).exists():
                z.write(d / f, f"songs/{yt_id}/{f}")
print(f"{out} ({out.stat().st_size/1e6:.0f} MB)")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/karaoke"); dest.mkdir(exist_ok=True)
    import shutil; shutil.copy(out, dest / name)
    print(f"copiado para o Drive: karaoke/{name}")
else:
    from google.colab import files
    files.download(str(out))


## Observações

- A primeira música demora mais (download do modelo, ~600 MB). Depois, cada música de 4 min leva
  cerca de 20 a 40 s na GPU T4.
- Se o download no YouTube falhar com mensagem de robô, use `COOKIES_FILE` (ver célula 2).
- O zip não é comprimido (vídeo já é comprimido); ~50 a 150 MB por música em 1080p.
- Rode a célula 4 quantas vezes quiser: ela só inclui as músicas da lista `results` da célula 3.
